# M08 — Phase-One Research Report

This notebook reproduces M03-M08 from official Federal Reserve GSW data, validates the cross-milestone research contract, and displays the consolidated report tables and figures. A standard CPU runtime is sufficient.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_ROOT = Path('/content/market_risk') if 'google.colab' in sys.modules else Path.cwd().resolve()
if 'google.colab' in sys.modules:
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Run the complete accepted workflow

This command downloads and audits GSW data, reruns M03-M07, validates their shared contracts, writes consolidated tables, and creates seven figures. Runtime files under `/content/market_risk/data` are temporary.

In [ ]:
completed = subprocess.run(
    [sys.executable, 'scripts/run_m08_research_report.py', '--output-root', 'data', '--run-prerequisites'],
    text=True, capture_output=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError('M08 pipeline failed')
print('M03-M08 pipeline completed successfully.')

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

report_path = max((PROJECT_ROOT / 'data' / 'audit').glob('m08_research_report_*.json'))
report = json.loads(report_path.read_text())
assert report['status'] == 'PASS', report['checks']
display(pd.DataFrame({'check': report['checks'].keys(), 'passed': report['checks'].values()}))
print('Valuation date:', report['valuation_date'])
print('Portfolio:', report['portfolio'])
print('Reproducibility:', report['reproducibility'])

In [ ]:
risk = pd.DataFrame(report['primary_750_day_risk'])
display(risk.pivot(index='method', columns=['measure', 'confidence_level'], values='value').round(2))
display(pd.DataFrame(report['historical_var_backtests'])[[
    'window_size', 'confidence_level', 'kupiec_observation_count',
    'kupiec_exception_count', 'kupiec_exception_rate', 'kupiec_p_value',
    'independence_p_value', 'conditional_coverage_p_value'
]])

In [ ]:
for figure_path in report['figures']:
    display(Image(filename=figure_path))

## Interpretation boundary

The report is a public-data physical-measure market-risk study. Historical Simulation alone has rolling backtest evidence. Parametric Normal and PCA Monte Carlo are current-estimate benchmarks. The results are not regulatory capital numbers, executable market quotes, or investment recommendations.